# Lab 52087: Amortized Inference


---

## Task 1 — Amortized GAN inversion and comparison

**1.1 Amortize inversion into an encoder**  
Amortize optimization-based inversion into a network: (i) implement or use **optimization-based inversion** (optimize $z$ so that $G(z)\approx x$). (ii) Train an **encoder** $E$ so that $E(x)$ approximates a good latent (e.g. on pairs $(x,z)$ with $x=G(z)$). At test time, $\hat z = E(x)$ and $\hat x = G(\hat z)$ is the amortized reconstruction.

**1.2 Compare three schemes and report metrics**  
Compare: **(a)** direct optimization-based inversion; **(b)** encoder-only $\hat z = E(x)$; **(c)** encoder + refinement (optimize starting from $E(x)$). Define **your own metrics** (e.g. PSNR, LPIPS) and run on a **small dataset** (e.g. random samples from $G$). Report which method is best and when encoder+refinement approaches direct inversion.

---

## Task 2 (Bonus) — Language-controlled encoding / text-guided generation

Use **text** to control encoding or generation (e.g. text-to-image or text-guided editing). For example: condition the encoder on a text prompt (e.g. CLIP) so that $E(x,\text{prompt})$ yields a latent that follows the prompt; or combine encoder with text-conditioned refinement. Goal: show that text can steer the latent or reconstruction in a meaningful way.



Setup

In [ ]:
# !pip -q install --upgrade huggingface_hub transformers torchvision

import os
import sys
import time
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import make_grid

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Load the pretrained StyleGAN generator \(G\)

We use the Hugging Face model `hajar001/stylegan2-ffhq-128`.


In [ ]:
from huggingface_hub import hf_hub_download

model_file = hf_hub_download(
    repo_id="hajar001/stylegan2-ffhq-128",
    filename="style_gan.py"
)
sys.path.insert(0, os.path.dirname(model_file))

from style_gan import StyleGAN

G = StyleGAN.from_pretrained("hajar001/stylegan2-ffhq-128").to(device).eval()

Z_DIM = 512  # latent dimension per model card

def G_generate(z, truncation_psi=0.7):
    x = G.generate(z, truncation_psi=truncation_psi)  # [-1,1]
    x = (x + 1) / 2
    return x.clamp(0, 1)

def G_forward_with_grad(latent, truncation_psi=0.7):
    """Generator forward pass that allows gradients (use in optimization/encoder training)."""
    if G.w_mean_samples == 0:
        G.update_w_mean()
    w = G.mapping(latent)
    if truncation_psi < 1.0:
        w = G.w_mean + truncation_psi * (w - G.w_mean)
    w_expanded = w.unsqueeze(1).expand(-1, G.synthesis.num_layers, -1)
    x = G.synthesis(w_expanded)
    x = (x + 1) / 2
    return x.clamp(0, 1)

def show_tensor_images(x, nrow=4, title=None):
    grid = make_grid(x.detach().cpu(), nrow=nrow)
    plt.figure(figsize=(8,8))
    if title: plt.title(title)
    plt.imshow(grid.permute(1,2,0))
    plt.axis("off")
    plt.show()

# sanity check
z = torch.randn(8, Z_DIM, device=device)
x = G_generate(z, truncation_psi=0.7)
show_tensor_images(x, nrow=4, title="Samples from G")


__Task 0.0 — Optimization-based inversion__

Create a synthetic target $x_\text{tgt}=G(z_0)$ and implement per-image inversion: optimize $z$ (e.g. from random init) so that $G(z)\approx x_\text{tgt}$. This is the baseline that you will later amortize into an encoder. (you may use your code from last lab)


In [ ]:
torch.manual_seed(0)
z0 = torch.randn(1, Z_DIM, device=device)
x_tgt = G_generate(z0, truncation_psi=0.7)

show_tensor_images(x_tgt, nrow=1, title="Target image x_tgt = G(z0)")


In [ ]:
OPTISTEPS = 500
lr_inver = 0.05
z_inverted = torch.randn_like(z0, requires_grad=True)
optimizer_inversion = torch.optim.Adam([z_inverted], lr=lr_inver)
loss_function_inversion = nn.MSELoss()
losses_opt = []
truncation_psi = 0.7
if G.w_mean_samples == 0:
    G.update_w_mean()
for step_index in tqdm(range(OPTISTEPS)):
    optimizer_inversion.zero_grad()
    w_inverted = G.mapping(z_inverted)
    if truncation_psi < 1.0:
        w_inverted = G.w_mean + truncation_psi * (w_inverted - G.w_mean)
    w_expanded = w_inverted.unsqueeze(1).expand(-1, G.synthesis.num_layers, -1)
    x_reconstructed = G.synthesis(w_expanded)
    x_reconstructed = (x_reconstructed + 1) / 2
    x_reconstructed = x_reconstructed.clamp(0, 1)
    loss_value = loss_function_inversion(x_reconstructed, x_tgt)
    loss_value.backward()
    optimizer_inversion.step()
    losses_opt.append(loss_value.item())
x_opt_reconstruction = G_generate(z_inverted.detach(), truncation_psi=0.7)
show_tensor_images(torch.cat([x_tgt, x_opt_reconstruction], dim=0), nrow=2, title="Target and optimization-based reconstruction")

In [ ]:
plt.figure()
plt.plot(losses_opt)
plt.xlabel("iteration")
plt.ylabel("MSE loss")
plt.title("Optimization-based inversion loss curve")
plt.show()


__Optional: invert a real face image__

You can try optimization-based inversion on a real photo (128×128): set `REAL_IMAGE_PATH` and run the cell. The target may be off the generator manifold, so reconstruction quality can vary.


__Task 1.1 — Encoder $E$ and training__

Define an encoder $E$ that maps image $x$ to latent $\hat z$. Train $E$ on synthetic pairs $(z, x)$ with $x=G(z)$ so that $E(x)\approx z$ (or so that $G(E(x))\approx x$). At test time, $\hat z = E(x)$ is a single forward pass.


In [ ]:
class Encoder(nn.Module):
    # CNN encoder: x (3x128x128) -> z (512)
    def __init__(self, z_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 512, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 512, 4, 1, 0), nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Linear(512, z_dim)

    def forward(self, x):
        h = self.net(x).flatten(1)
        return self.fc(h)

# E = Encoder(Z_DIM).to(device)

# x_tmp = G_generate(torch.randn(2, Z_DIM, device=device))
# print("Shapes:", x_tmp.shape, E(x_tmp).shape)


In [ ]:
batch_size_encoder = 32
TRAINSTEPSENCOD = 2000
learning_rate_encoder = 1e-4
encoder_model = Encoder(Z_DIM).to(device)
optimizer_encoder = torch.optim.Adam(encoder_model.parameters(), lr=learning_rate_encoder)
loss_function_encoder = nn.MSELoss()
encoder_model.train()
encoder_losses = []
for step_index in tqdm(range(TRAINSTEPSENCOD)):
    latent_batch = torch.randn(batch_size_encoder, Z_DIM, device=device)
    image_batch = G_generate(latent_batch, truncation_psi=0.7)
    predicted_latent_batch = encoder_model(image_batch)
    reconstructed_batch = G_forward_with_grad(predicted_latent_batch, truncation_psi=0.7)
    loss_value = loss_function_encoder(reconstructed_batch, image_batch)
    optimizer_encoder.zero_grad()
    loss_value.backward()
    optimizer_encoder.step()
    encoder_losses.append(loss_value.item())
plt.figure()
plt.plot(encoder_losses, alpha=0.4, label="raw")
plt.plot(np.convolve(encoder_losses, np.ones(50) / 50, mode="valid"), color="C0", lw=2, label="smoothed (50)")
plt.xlabel("iteration")
plt.ylabel("reconstruction MSE loss")
plt.title("Encoder training loss (image-space reconstruction)")
plt.legend()
plt.show()
encoder_model.eval()

__Task 1.2 — Compare three schemes and report metrics__

Implement a comparison of: **(a)** direct optimization-based inversion; **(b)** encoder-only $\hat z = E(x)$; **(c)** encoder + refinement (few optimization steps from $E(x)$).  
Define **your own evaluation metrics** (e.g. PSNR, LPIPS, SSIM, or MSE in pixel/latent space) and evaluate on a **small dataset** (e.g. 16–64 random samples from $G$). Report a short summary or table: which method gives the best reconstruction on your metrics, and how does encoder+refinement compare to direct inversion as you vary the number of refinement steps?


In [ ]:
def compute_mse(image_batch_prediction, image_batch_target):
    return F.mse_loss(image_batch_prediction, image_batch_target, reduction="mean").item()

def compute_psnr(image_batch_prediction, image_batch_target):
    mse_value = F.mse_loss(image_batch_prediction, image_batch_target, reduction="mean").item()
    if mse_value == 0:
        return float("inf")
    return 10.0 * np.log10(1.0 / mse_value)

def invert_image_optimization(target_image, num_steps, learning_rate):
    latent_variable = torch.randn(1, Z_DIM, device=device, requires_grad=True)
    optimizer_latent = torch.optim.Adam([latent_variable], lr=learning_rate)
    loss_function = nn.MSELoss()
    for step_index in range(num_steps):
        optimizer_latent.zero_grad()
        reconstructed_image = G_forward_with_grad(latent_variable, truncation_psi=0.7)
        loss_value = loss_function(reconstructed_image, target_image)
        loss_value.backward()
        optimizer_latent.step()
    return latent_variable.detach()

def refine_from_encoder(target_image, initial_latent, num_steps, learning_rate):
    latent_variable = initial_latent.clone().detach().requires_grad_(True)
    optimizer_latent = torch.optim.Adam([latent_variable], lr=learning_rate)
    loss_function = nn.MSELoss()
    for step_index in range(num_steps):
        optimizer_latent.zero_grad()
        reconstructed_image = G_forward_with_grad(latent_variable, truncation_psi=0.7)
        loss_value = loss_function(reconstructed_image, target_image)
        loss_value.backward()
        optimizer_latent.step()
    return latent_variable.detach()

num_evaluation_samples = 32
num_optimization_steps_direct = 300
num_refinement_steps = 100
learning_rate_direct = 0.05
learning_rate_refinement = 0.02
mse_direct_list = []
psnr_direct_list = []
mse_encoder_list = []
psnr_encoder_list = []
mse_refined_list = []
psnr_refined_list = []
with torch.no_grad():
    latent_dataset = torch.randn(num_evaluation_samples, Z_DIM, device=device)
    image_dataset = G_generate(latent_dataset, truncation_psi=0.7)
for sample_index in range(num_evaluation_samples):
    target_image = image_dataset[sample_index : sample_index + 1]
    latent_direct = invert_image_optimization(target_image, num_optimization_steps_direct, learning_rate_direct)
    reconstruction_direct = G_generate(latent_direct, truncation_psi=0.7).detach()
    with torch.no_grad():
        latent_encoder = encoder_model(target_image)
        reconstruction_encoder = G_generate(latent_encoder, truncation_psi=0.7)
    latent_refined = refine_from_encoder(target_image, latent_encoder, num_refinement_steps, learning_rate_refinement)
    reconstruction_refined = G_generate(latent_refined, truncation_psi=0.7).detach()
    mse_direct_list.append(compute_mse(reconstruction_direct, target_image))
    psnr_direct_list.append(compute_psnr(reconstruction_direct, target_image))
    mse_encoder_list.append(compute_mse(reconstruction_encoder, target_image))
    psnr_encoder_list.append(compute_psnr(reconstruction_encoder, target_image))
    mse_refined_list.append(compute_mse(reconstruction_refined, target_image))
    psnr_refined_list.append(compute_psnr(reconstruction_refined, target_image))
print("Direct optimization MSE:", np.mean(mse_direct_list))
print("Direct optimization PSNR:", np.mean(psnr_direct_list))
print("Encoder-only MSE:", np.mean(mse_encoder_list))
print("Encoder-only PSNR:", np.mean(psnr_encoder_list))
print("Encoder + refinement MSE:", np.mean(mse_refined_list))
print("Encoder + refinement PSNR:", np.mean(psnr_refined_list))
example_indices = torch.arange(0, min(8, num_evaluation_samples))
with torch.no_grad():
    example_images = image_dataset[example_indices]
    example_latent_encoder = encoder_model(example_images)
    example_reconstruction_encoder = G_generate(example_latent_encoder, truncation_psi=0.7)
show_tensor_images(example_images, nrow=4, title="Evaluation targets")
show_tensor_images(example_reconstruction_encoder, nrow=4, title="Encoder-only reconstructions")

__Task 2 (Bonus) — Language-controlled encoding / text-guided generation__

Use **text** to control the encoding or generation process so that you can achieve a form of **text-to-image** or **text-guided face editing**. For example: condition the encoder on a text prompt (e.g. via CLIP text embeddings) so that $E(x, \text{prompt})$ produces a latent that reconstructs or edits the face according to the prompt; or combine an encoder with a text-conditioned refinement step. You may use the provided CLIP-based setup or consider stronger / different models. The goal is to demonstrate that text can steer the latent or the reconstruction in a meaningful way.


In [ ]:
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def preprocess_for_clip(image_tensor):
    image_tensor_clamped = image_tensor.clamp(0.0, 1.0)
    image_batch = image_tensor_clamped.detach().cpu()
    image_list = [transforms.ToPILImage()(image_batch[i]) for i in range(image_batch.size(0))]
    inputs = clip_processor(images=image_list, return_tensors="pt", padding=True)
    return {key: value.to(device) for key, value in inputs.items()}

def generate_image_from_text(prompt_text, num_steps, learning_rate):
    text_inputs = clip_processor(text=[prompt_text], return_tensors="pt", padding=True)
    text_inputs = {key: value.to(device) for key, value in text_inputs.items()}
    with torch.no_grad():
        text_features = clip_model.get_text_features(**text_inputs)
    latent_variable = torch.randn(1, Z_DIM, device=device, requires_grad=True)
    optimizer_latent = torch.optim.Adam([latent_variable], lr=learning_rate)
    for step_index in range(num_steps):
        optimizer_latent.zero_grad()
        generated_image = G_generate(latent_variable, truncation_psi=0.7)
        image_inputs = preprocess_for_clip(generated_image)
        image_features = clip_model.get_image_features(**image_inputs)
        image_features_normalized = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features_normalized = text_features / text_features.norm(dim=-1, keepdim=True)
        similarity = (image_features_normalized * text_features_normalized).sum()
        loss_value = -similarity
        loss_value.backward()
        optimizer_latent.step()
    final_image = G_generate(latent_variable.detach(), truncation_psi=0.7)
    return final_image

text_prompt_example = "a smiling person"
generated_image_from_text = generate_image_from_text(text_prompt_example, num_steps=200, learning_rate=0.05)
show_tensor_images(generated_image_from_text, nrow=1, title=f"Text-guided generation: {text_prompt_example}")

__Task3 (Not bonus but just try it out a bit, I won't score this one): try to have fun with the Dust3r__
E.g. reconstruct 3D from two images and show the 3D.

In [ ]:
try:
    import dust3r
    print("Dust3r is available. Please provide image paths and run your own 3D reconstruction experiments.")
except ImportError:
    print("Dust3r is not installed in this environment.")